
<div align="center">

# BFF5525 Project
## **Semester 1 2026**

</div>

# Instructions

- This is an individual project designed to assess all six learning objectives of the unit.
- The project is marked out of 25 marks and contributes 25% to your overall unit grade.
- **Due date:** 11:55pm, Monday 18 May 2026  

- Late submissions without an approved extension will incur a penalty of 5% per day.

---

### Support

For assistance:

- Post questions on the assessment forum on Moodle  
- Attend weekly consultation sessions

---

## AI Usage Policy

You are required to use the `pyfin` libraries where relevant. 

Where you need to use codes outside `pyfin`, AI tools (e.g., ChatGPT, Copilot) may be used to assist with code generation and debugging only  
AI must not be used to generate answers to interpretation, verification, or conceptual questions.

You are fully responsible for the correctness of any AI-assisted code and for ensuring that all written responses reflect your own understanding and are consistent with your computed results. Responses that are generic or inconsistent with your outputs may receive reduced marks.

Complete the Declaration at the end of this notebook.


## Project Overview

You are given a fixed dataset of 50 large-cap US stocks (monthly returns) in file `screened_50stocks_monthly_returns.csv` and industry information in file `screened_50stocks_industry.csv`


Treat this as a screened investment universe.

---

### Core Task

You will:

1. Construct baseline portfolios using methods covered in the unit  
2. Identify one key issue in the baseline results  
3. Propose and implement one modification to address that issue  
4. Evaluate how this modification affects:
   - portfolio weights  
   - performance  
   - reliability  

---

### Core Question to address

> How much confidence should we place in the results of a portfolio construction method?

---



## Submission 

- Files to submit:
  - This notebook, renamed as `Project_YourStudentID`
  - Your `pyfin` folder (as a `.zip` file) which includes additional reuseable functions

- Upload your files to the Moodle project submission box  


+ Your notebook must:

    - contain all code, output, tables, figures, and discussion  
    - show full output  
    - have cells executed in order from top to bottom  
    - include brief commentary throughout (not just code)  
    - include the final report at the end (markdown cells)  

Do not submit a separate report file.

## Required Structure

Organise your notebook as follows:

1. **Setup**  
   Data loading, imports, helper functions if any  

2. **Baseline Portfolios**  
   Construct and examine initial portfolios  

3. **Modified Portfolios**  
   Identify and address issues in the baseline  

4. **Benchmark**  
   Propose, construct, and justify your benchmark. Compare the performance of baseline and modified portfolios against this benchmark.  

5. **Reliability Analysis**  
   Validation, robustness, model risk on your best looking portfolio 

6. **Final Report**  
   Summary and interpretation



### Commentary Requirement

You should include brief, clear commentary throughout your notebook:

- explain what you are doing  
- highlight key observations  
- interpret results as they appear  

Avoid long blocks of code without explanation.

## Portfolio Construction

### Baseline

Construct baseline portfolios using methods covered in the unit.

Examine the results carefully:

- Are allocations economically plausible?  
- Is there concentration (stocks or industries)?  
- Are there signs of instability?

---

### Modification

> Identify one main issue in the baseline portfolios.

Then:

- propose one modification to address it  
- justify your choice  
- explain what you expect to change  

Your modification should be:

- economically meaningful  
- clearly motivated by your observations    

## Benchmark Requirement

You must propose and construct one benchmark portfolio.

Your benchmark should:

- be simple and investable  
- not rely on optimisation  
- be clearly justified  

You should use it to evaluate your portfolios.



## Reliability Analysis

Focus on the best-looking portfolio from your earlier analysis.

You must evaluate its reliability using:

### Validation
- Does the result hold across different time periods?

### Robustness
- Does the result change under reasonable variations?

### Model Risk
- Does your conclusion depend on modelling choices?

---

### Important

Do not test every portfolio.

> Focus on whether your chosen portfolio remains convincing under scrutiny.

## Final Report

Include a concise report at the end of your notebook.

### Suggested length
500–800 words

---

### Your report should include:

- Summary of findings  
- Interpretation of results  
- Reliability assessment  
- Limitations of the analysis


## Implementation Notes

- Show all outputs clearly
- Commentary and report are to be in Markdown cells
- Keep code organised and readable  
- Add reusable functions to `pyfin` where appropriate  

Avoid unnecessary complexity.

## Marking Rubric (25%)

### 1. Implementation & Organisation (5)
- clear, reproducible notebook  
- appropriate construction of portfolios  
- effective use of inline commentary  



### 2. Baseline & Modification Analysis (6)
- clear identification of a meaningful issue  
- well-justified modification  
- insightful comparison of baseline vs modified  


### 3. Benchmark (3)
- appropriate and justified  
- used meaningfully in evaluation  


### 4. Reliability Analysis (6)
- focuses on the best-looking portfolio  
- includes validation, robustness, model risk  
- explains what changes and why  


### 5. Final Report (5)
- concise and well-structured  
- balanced interpretation  
- includes meaningful discussion of limitations  
- avoids overstating conclusions  


### Key expectation

Strong projects:

- show clear reasoning  
- connect results to interpretation  
- recognise uncertainty and limitations  


## AI Declaration (required)
Delete the option that does not apply.

- ☐ Yes, I used AI for code/debugging
- ☐ No, I did not use AI.

In [22]:
# ============================================================
# 1. Setup and Data Understanding
# ============================================================

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# Load pyfin helper functions
sys.path.insert(0, "../pyfin")

from pyfin import (
    annualised_ret,
    annualised_vol,
    rolling_vol,
    max_drawdown,
    VaR_historical,
    VaR_normal,
    CVaR_historical,
    sharpe_ratio,
    tear_sheet
)

# Load data
monthly_ret_raw = pd.read_csv("../data/screened_50stocks_monthly_returns.csv")
industry_map = pd.read_csv("../data/screened_50stocks_industry.csv")

# Convert Date, sort observations, and set Date as index
ret = monthly_ret_raw.copy()
ret["Date"] = pd.to_datetime(ret["Date"], errors="coerce")
ret = ret.sort_values("Date").set_index("Date")

# Basic checks
data_checks = pd.Series({
    "start_date": ret.index.min(),
    "end_date": ret.index.max(),
    "n_months": ret.shape[0],
    "n_stocks": ret.shape[1],
    "dates_sorted": ret.index.is_monotonic_increasing,
    "duplicated_dates": ret.index.duplicated().sum(),
    "total_missing_values": ret.isna().sum().sum(),
    "stocks_with_missing_values": (ret.isna().sum() > 0).sum(),
    "duplicated_tickers_in_industry": industry_map["ticker"].duplicated().sum(),
    "stocks_without_industry_label": len(set(ret.columns) - set(industry_map["ticker"])),
    "industry_tickers_not_in_returns": len(set(industry_map["ticker"]) - set(ret.columns))
})

display(data_checks)

# Industry distribution
display(industry_map["industry"].value_counts())

# Extreme return check: monthly return larger than +/-50%
extreme_returns = (
    ret[ret.abs() > 0.50]
    .stack()
    .reset_index()
)

extreme_returns.columns = ["Date", "ticker", "return"]

display(extreme_returns)

start_date                         2010-01-01 00:00:00
end_date                           2025-12-01 00:00:00
n_months                                           192
n_stocks                                            50
dates_sorted                                      True
duplicated_dates                                     0
total_missing_values                                 0
stocks_with_missing_values                           0
duplicated_tickers_in_industry                       0
stocks_without_industry_label                        0
industry_tickers_not_in_returns                      0
dtype: object

industry
Tech             10
Financials        8
Healthcare        8
Consumer          8
Industrials       6
Energy            4
Communication     3
Utilities         3
Name: count, dtype: int64

,Date,ticker,return
0,2010-01-01,AAPL,NaN
1,2010-01-01,MCD,NaN
2,2010-01-01,MRK,NaN
3,2010-01-01,MS,NaN
4,2010-01-01,MSFT,NaN
...,...,...,...
9595,2025-12-01,GS,NaN
9596,2025-12-01,HD,NaN
9597,2025-12-01,HON,NaN
9598,2025-12-01,CSCO,NaN
